In [1]:
import sys
!{sys.executable} -m pip install xgboost psycopg2-binary sqlalchemy faker

In [2]:
import importlib
import subprocess

packages = ['xgboost', 'psycopg2', 'sqlalchemy', 'faker']
for pkg in packages:
    try:
        importlib.import_module(pkg)
        print(f"{pkg} ✅")
    except ImportError:
        print(f"{pkg} ❌ still not found")

xgboost ✅
psycopg2 ✅
sqlalchemy ✅
faker ✅


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from faker import Faker
import sklearn
import xgboost
import psycopg2
import sqlalchemy

print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("sklearn:", sklearn.__version__)
print("xgboost:", xgboost.__version__)
print("All libraries loaded successfully ✅")

pandas: 2.2.3
numpy: 2.1.3
sklearn: 1.6.1
xgboost: 3.2.0
All libraries loaded successfully ✅


## **Data Generation**

In [4]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime, timedelta
import os
import warnings
warnings.filterwarnings('ignore')

fake = Faker('en_IN')  # Indian locale
np.random.seed(42)
random.seed(42)

# ── Paths ──────────────────────────────────────────────────────────────────
RAW = r"D:\Projects\End-to-end projects\11. Fashion Trend Inventory\Data\Raw"

print("Setup complete ✅")
print(f"Saving data to: {RAW}")

Setup complete ✅
Saving data to: D:\Projects\End-to-end projects\11. Fashion Trend Inventory\Data\Raw


In [5]:
# ── Product Categories ─────────────────────────────────────────────────────
categories = {
    'Topwear':      ['Oversized T-Shirt', 'Crop Top', 'Shirt', 'Hoodie', 'Sweatshirt', 'Polo T-Shirt', 'Tank Top', 'Blazer'],
    'Bottomwear':   ['Baggy Jeans', 'Cargo Pants', 'Joggers', 'Mini Skirt', 'Palazzo', 'Straight Fit Jeans', 'Shorts'],
    'Ethnic':       ['Kurta Set', 'Co-ord Saree', 'Anarkali', 'Sharara Set', 'Dhoti Pants', 'Nehru Jacket'],
    'Dresses':      ['Maxi Dress', 'Bodycon Dress', 'Wrap Dress', 'Shirt Dress', 'Co-ord Set'],
    'Footwear':     ['Chunky Sneakers', 'Kolhapuri', 'Block Heels', 'Loafers', 'Sneakers', 'Sandals'],
    'Accessories':  ['Sling Bag', 'Tote Bag', 'Bucket Hat', 'Caps', 'Sunglasses', 'Belts', 'Scarves']
}

# ── Trend Names (real 2023-2025 Indian fashion trends) ─────────────────────
trend_names = [
    'Cottagecore', 'Y2K Revival', 'Quiet Luxury', 'Gorpcore', 'Dopamine Dressing',
    'Barbiecore', 'Dark Academia', 'Coastal Grandmother', 'Mob Wife Aesthetic',
    'Coquette', 'Old Money', 'Streetwear Fusion', 'Ethnic Modernism', 'Athleisure 2.0',
    'Maximalism', 'Clean Girl Aesthetic', 'Cottagecore Ethnic', 'Gender Fluid Fashion',
    'Vintage Revival', 'Sustainable Chic'
]

# ── Indian Cities with Tier ────────────────────────────────────────────────
cities = {
    'Mumbai':     {'tier': 1, 'region': 'West',  'trend_sensitivity': 0.95},
    'Delhi':      {'tier': 1, 'region': 'North', 'trend_sensitivity': 0.93},
    'Bangalore':  {'tier': 1, 'region': 'South', 'trend_sensitivity': 0.92},
    'Hyderabad':  {'tier': 1, 'region': 'South', 'trend_sensitivity': 0.88},
    'Chennai':    {'tier': 1, 'region': 'South', 'trend_sensitivity': 0.85},
    'Kolkata':    {'tier': 1, 'region': 'East',  'trend_sensitivity': 0.83},
    'Pune':       {'tier': 2, 'region': 'West',  'trend_sensitivity': 0.80},
    'Ahmedabad':  {'tier': 2, 'region': 'West',  'trend_sensitivity': 0.78},
    'Jaipur':     {'tier': 2, 'region': 'North', 'trend_sensitivity': 0.75},
    'Lucknow':    {'tier': 2, 'region': 'North', 'trend_sensitivity': 0.72},
    'Surat':      {'tier': 2, 'region': 'West',  'trend_sensitivity': 0.70},
    'Chandigarh': {'tier': 2, 'region': 'North', 'trend_sensitivity': 0.74},
    'Indore':     {'tier': 3, 'region': 'Central','trend_sensitivity': 0.60},
    'Bhopal':     {'tier': 3, 'region': 'Central','trend_sensitivity': 0.58},
    'Nagpur':     {'tier': 3, 'region': 'West',  'trend_sensitivity': 0.55},
    'Coimbatore': {'tier': 3, 'region': 'South', 'trend_sensitivity': 0.57},
    'Patna':      {'tier': 3, 'region': 'East',  'trend_sensitivity': 0.50},
    'Kochi':      {'tier': 3, 'region': 'South', 'trend_sensitivity': 0.62}
}

# ── Brands ─────────────────────────────────────────────────────────────────
brands = {
    'Bewakoof':   {'segment': 'Value',   'base_price_multiplier': 1.0},
    'Snitch':     {'segment': 'Premium', 'base_price_multiplier': 1.6},
    'The Souled Store': {'segment': 'Value', 'base_price_multiplier': 1.1},
    'Urbanic':    {'segment': 'Premium', 'base_price_multiplier': 1.7},
    'FabAlley':   {'segment': 'Premium', 'base_price_multiplier': 1.5},
    'Libas':      {'segment': 'Ethnic',  'base_price_multiplier': 1.3},
    'Zudio':      {'segment': 'Budget',  'base_price_multiplier': 0.7},
    'H&M India':  {'segment': 'Premium', 'base_price_multiplier': 1.8},
    'Myntra M':   {'segment': 'Value',   'base_price_multiplier': 1.2},
    'Styli':      {'segment': 'Premium', 'base_price_multiplier': 1.6}
}

# ── Seasons ────────────────────────────────────────────────────────────────
seasons = {
    'Summer':  {'months': [3,4,5],    'peak_month': 4},
    'Monsoon': {'months': [6,7,8],    'peak_month': 7},
    'Festive': {'months': [9,10,11],  'peak_month': 10},
    'Winter':  {'months': [11,12,1],  'peak_month': 12}
}

print(f"Categories: {sum(len(v) for v in categories.values())} SKU types")
print(f"Trends: {len(trend_names)}")
print(f"Cities: {len(cities)}")
print(f"Brands: {len(brands)}")
print("Dimensions loaded ✅")

Categories: 39 SKU types
Trends: 20
Cities: 18
Brands: 10
Dimensions loaded ✅


In [6]:
def get_season(month):
    for s, v in seasons.items():
        if month in v['months']:
            return s
    return 'Winter'

def base_price(category, brand_name):
    price_map = {
        'Topwear': 799, 'Bottomwear': 999, 'Ethnic': 1499,
        'Dresses': 1299, 'Footwear': 1799, 'Accessories': 599
    }
    multiplier = brands[brand_name]['base_price_multiplier']
    base = price_map.get(category, 999)
    return round(base * multiplier * np.random.uniform(0.85, 1.15), -1)

products = []
sku_id = 1000

for brand_name in brands:
    for category, subcats in categories.items():
        for subcat in subcats:
            for color in random.sample(['Black','White','Beige','Olive','Navy','Rust','Pink','Yellow','Brown','Grey'], 3):
                price = base_price(category, brand_name)
                cost  = round(price * np.random.uniform(0.35, 0.50), -1)
                products.append({
                    'sku_id':           f'SKU{sku_id:05d}',
                    'product_name':     f'{color} {subcat}',
                    'category':         category,
                    'subcategory':      subcat,
                    'brand':            brand_name,
                    'color':            color,
                    'segment':          brands[brand_name]['segment'],
                    'mrp':              price,
                    'cost_price':       cost,
                    'gross_margin_pct': round((price - cost) / price * 100, 2),
                    'launch_season':    random.choice(list(seasons.keys())),
                    'is_active':        random.choices([1, 0], weights=[0.85, 0.15])[0]
                })
                sku_id += 1

df_products = pd.DataFrame(products)
df_products.to_csv(os.path.join(RAW, 'products.csv'), index=False)

print(f"Products table: {df_products.shape[0]:,} rows × {df_products.shape[1]} columns ✅")
df_products.head(3)

Products table: 1,170 rows × 12 columns ✅


,sku_id,product_name,category,subcategory,brand,color,segment,mrp,cost_price,gross_margin_pct,launch_season,is_active
0,SKU01000,White Oversized T-Shirt,Topwear,Oversized T-Shirt,Bewakoof,White,Value,770.0,380.0,50.65,Monsoon,1
1,SKU01001,Black Oversized T-Shirt,Topwear,Oversized T-Shirt,Bewakoof,Black,Value,850.0,370.0,56.47,Summer,1
2,SKU01002,Navy Oversized T-Shirt,Topwear,Oversized T-Shirt,Bewakoof,Navy,Value,720.0,270.0,62.50,Summer,1


In [7]:
def trend_lifecycle(week_num, peak_week, total_weeks=104):
    """Simulates a realistic trend rise-peak-decay curve"""
    x = (week_num - peak_week) / (total_weeks * 0.15)
    score = np.exp(-0.5 * x**2)  # Gaussian bell curve
    noise = np.random.uniform(-0.05, 0.05)
    return round(min(max(score + noise, 0.01), 1.0), 4)

start_date = datetime(2023, 1, 1)
trend_records = []

for trend in trend_names:
    peak_week    = random.randint(15, 90)
    max_score    = round(random.uniform(0.55, 1.0), 2)
    decay_speed  = random.choice(['fast', 'medium', 'slow'])
    decay_map    = {'fast': 0.20, 'medium': 0.12, 'slow': 0.06}
    decay_rate   = decay_map[decay_speed]

    # Map trend to relevant categories
    if trend in ['Ethnic Modernism', 'Cottagecore Ethnic']:
        rel_cats = ['Ethnic', 'Accessories']
    elif trend in ['Athleisure 2.0', 'Gorpcore']:
        rel_cats = ['Topwear', 'Bottomwear', 'Footwear']
    elif trend in ['Y2K Revival', 'Barbiecore', 'Coquette']:
        rel_cats = ['Dresses', 'Accessories', 'Topwear']
    elif trend in ['Quiet Luxury', 'Old Money', 'Clean Girl Aesthetic']:
        rel_cats = ['Topwear', 'Bottomwear', 'Accessories']
    else:
        rel_cats = random.sample(list(categories.keys()), 3)

    for week_num in range(104):   # 2 years of weekly data
        week_date   = start_date + timedelta(weeks=week_num)
        raw_score   = trend_lifecycle(week_num, peak_week)
        trend_score = round(raw_score * max_score, 4)

        # Search volume index (proxy for Google Trends)
        search_vol  = int(trend_score * random.randint(80000, 150000))

        # Social mentions (Instagram + Pinterest proxy)
        social_mentions = int(trend_score * random.randint(5000, 25000))

        # Trend phase
        if week_num < peak_week * 0.4:
            phase = 'Emerging'
        elif week_num < peak_week * 0.8:
            phase = 'Rising'
        elif week_num < peak_week * 1.2:
            phase = 'Peaking'
        elif week_num < peak_week * 1.8:
            phase = 'Declining'
        else:
            phase = 'Dead'

        trend_records.append({
            'trend_id':           f'TRD{trend_names.index(trend)+1:03d}',
            'trend_name':         trend,
            'week_start_date':    week_date.strftime('%Y-%m-%d'),
            'week_number':        week_num + 1,
            'year':               week_date.year,
            'month':              week_date.month,
            'trend_score':        trend_score,
            'search_volume_idx':  search_vol,
            'social_mentions':    social_mentions,
            'trend_phase':        phase,
            'peak_week':          peak_week,
            'decay_speed':        decay_speed,
            'relevant_categories': '|'.join(rel_cats),
            'max_score':          max_score
        })

df_trends = pd.DataFrame(trend_records)
df_trends.to_csv(os.path.join(RAW, 'trend_signals.csv'), index=False)

print(f"Trend Signals table: {df_trends.shape[0]:,} rows × {df_trends.shape[1]} columns ✅")
df_trends.head(3)

Trend Signals table: 2,080 rows × 14 columns ✅


,trend_id,trend_name,week_start_date,week_number,year,month,trend_score,search_volume_idx,social_mentions,trend_phase,peak_week,decay_speed,relevant_categories,max_score
0,TRD001,Cottagecore,2023-01-01,1,2023,1,0.0218,2946,181,Emerging,75,fast,Accessories|Topwear|Ethnic,0.87
1,TRD001,Cottagecore,2023-01-08,2,2023,1,0.0087,1289,120,Emerging,75,fast,Accessories|Topwear|Ethnic,0.87
2,TRD001,Cottagecore,2023-01-15,3,2023,1,0.0087,852,162,Emerging,75,fast,Accessories|Topwear|Ethnic,0.87


In [8]:
inventory_records = []
inventory_dates = pd.date_range(start='2023-01-01', end='2024-12-31', freq='W-MON')

for _, sku in df_products.iterrows():
    # Opening stock depends on segment
    seg_stock = {'Budget': 2000, 'Value': 1500, 'Premium': 800, 'Ethnic': 1000}
    opening   = seg_stock.get(sku['segment'], 1000) + random.randint(-200, 200)

    current_stock = opening
    reorder_point = int(opening * 0.25)
    lead_time_days = random.randint(30, 75)  # Indian supplier lead times

    for i, snap_date in enumerate(inventory_dates):
        # Weekly sales velocity (units sold this week)
        base_velocity = random.randint(10, 80)

        # Seasonal boost
        month = snap_date.month
        season = get_season(month)
        if season == sku['launch_season']:
            velocity = int(base_velocity * random.uniform(1.3, 2.0))
        else:
            velocity = int(base_velocity * random.uniform(0.4, 0.9))

        units_sold    = min(velocity, current_stock)
        units_received = 0

        # Reorder logic
        if current_stock <= reorder_point:
            units_received = int(opening * random.uniform(0.5, 0.8))

        closing_stock = current_stock - units_sold + units_received
        closing_stock = max(closing_stock, 0)

        # Stockout flag
        stockout_flag = 1 if current_stock == 0 else 0

        # Inventory health
        weeks_of_cover = round(closing_stock / max(velocity, 1), 1)
        if weeks_of_cover > 16:
            inv_health = 'Overstock'
        elif weeks_of_cover > 8:
            inv_health = 'Healthy'
        elif weeks_of_cover > 3:
            inv_health = 'Low'
        else:
            inv_health = 'Critical'

        inventory_records.append({
            'inventory_id':     f'INV{len(inventory_records)+1:07d}',
            'sku_id':           sku['sku_id'],
            'brand':            sku['brand'],
            'category':         sku['category'],
            'snapshot_date':    snap_date.strftime('%Y-%m-%d'),
            'week_number':      i + 1,
            'opening_stock':    current_stock,
            'units_sold':       units_sold,
            'units_received':   units_received,
            'closing_stock':    closing_stock,
            'reorder_point':    reorder_point,
            'lead_time_days':   lead_time_days,
            'stockout_flag':    stockout_flag,
            'weeks_of_cover':   weeks_of_cover,
            'inventory_health': inv_health,
            'inventory_value':  round(closing_stock * sku['cost_price'], 2)
        })

        current_stock = closing_stock

df_inventory = pd.DataFrame(inventory_records)
df_inventory.to_csv(os.path.join(RAW, 'inventory.csv'), index=False)

print(f"Inventory table: {df_inventory.shape[0]:,} rows × {df_inventory.shape[1]} columns ✅")
df_inventory.head(3)

Inventory table: 122,850 rows × 16 columns ✅


,inventory_id,sku_id,brand,category,snapshot_date,week_number,opening_stock,units_sold,units_received,closing_stock,reorder_point,lead_time_days,stockout_flag,weeks_of_cover,inventory_health,inventory_value
0,INV0000001,SKU01000,Bewakoof,Topwear,2023-01-02,1,1573,30,0,1543,393,68,0,51.4,Overstock,586340.0
1,INV0000002,SKU01000,Bewakoof,Topwear,2023-01-09,2,1543,9,0,1534,393,68,0,170.4,Overstock,582920.0
2,INV0000003,SKU01000,Bewakoof,Topwear,2023-01-16,3,1534,6,0,1528,393,68,0,254.7,Overstock,580640.0


In [9]:
sales_records = []
sale_id = 1

for _ in range(120000):
    sku        = df_products.sample(1).iloc[0]
    city_name  = random.choices(list(cities.keys()),
                     weights=[1/cities[c]['tier']**1.5 for c in cities])[0]
    city_info  = cities[city_name]

    sale_date  = fake.date_between(start_date=datetime(2023,1,1),
                                   end_date=datetime(2024,12,31))
    month      = sale_date.month
    season     = get_season(month)

    # Price after discount
    discount_pct = random.choices(
        [0, 5, 10, 15, 20, 25, 30, 40, 50],
        weights=[20, 15, 20, 15, 10, 8, 6, 4, 2]
    )[0]
    selling_price = round(sku['mrp'] * (1 - discount_pct/100), 2)
    quantity      = random.choices([1,2,3,4,5], weights=[50,25,12,8,5])[0]
    revenue       = round(selling_price * quantity, 2)
    cogs          = round(sku['cost_price'] * quantity, 2)
    gross_profit  = round(revenue - cogs, 2)

    # Channel
    channel = random.choices(
        ['D2C Website', 'Myntra', 'Ajio', 'Amazon', 'Offline Store', 'Meesho'],
        weights=[25, 22, 18, 15, 12, 8]
    )[0]

    # Return flag (size issues drive returns)
    return_prob = 0.28 if sku['category'] in ['Topwear','Bottomwear','Dresses'] else 0.12
    if discount_pct == 0:
        return_prob *= 0.8
    is_returned = random.choices([1,0], weights=[return_prob, 1-return_prob])[0]

    sales_records.append({
        'sale_id':          f'SAL{sale_id:07d}',
        'sku_id':           sku['sku_id'],
        'brand':            sku['brand'],
        'category':         sku['category'],
        'subcategory':      sku['subcategory'],
        'sale_date':        sale_date.strftime('%Y-%m-%d'),
        'month':            month,
        'year':             sale_date.year,
        'season':           season,
        'city':             city_name,
        'region':           city_info['region'],
        'city_tier':        city_info['tier'],
        'channel':          channel,
        'quantity':         quantity,
        'mrp':              sku['mrp'],
        'discount_pct':     discount_pct,
        'selling_price':    selling_price,
        'revenue':          revenue,
        'cogs':             cogs,
        'gross_profit':     gross_profit,
        'gross_margin_pct': round(gross_profit/revenue*100, 2) if revenue > 0 else 0,
        'is_returned':      is_returned,
        'trend_sensitivity':city_info['trend_sensitivity']
    })
    sale_id += 1

df_sales = pd.DataFrame(sales_records)
df_sales.to_csv(os.path.join(RAW, 'sales_transactions.csv'), index=False)

print(f"Sales table: {df_sales.shape[0]:,} rows × {df_sales.shape[1]} columns ✅")
df_sales.head(3)

Sales table: 120,000 rows × 23 columns ✅


,sale_id,sku_id,brand,category,subcategory,sale_date,month,year,season,city,...,quantity,mrp,discount_pct,selling_price,revenue,cogs,gross_profit,gross_margin_pct,is_returned,trend_sensitivity
0,SAL0000001,SKU01394,Urbanic,Bottomwear,Shorts,2024-06-06,6,2024,Monsoon,Kolkata,...,1,1680.0,5,1596.0,1596.0,830.0,766.0,47.99,1,0.83
1,SAL0000002,SKU02142,Styli,Footwear,Loafers,2023-01-13,1,2023,Winter,Jaipur,...,5,2570.0,5,2441.5,12207.5,5050.0,7157.5,58.63,0,0.75
2,SAL0000003,SKU01119,Snitch,Topwear,Oversized T-Shirt,2024-04-22,4,2024,Summer,Lucknow,...,1,1330.0,5,1263.5,1263.5,500.0,763.5,60.43,0,0.72


In [11]:
po_records = []
po_id = 1

suppliers = ['Tiruppur Mills', 'Surat Fabrics Co', 'Delhi Garment Hub',
             'Ludhiana Knits', 'Bangalore Weaves', 'Jaipur Ethnic Works',
             'Mumbai Exports Ltd', 'Chennai Apparel Co']

for _ in range(8000):
    sku       = df_products.sample(1).iloc[0]
    po_date   = fake.date_between(start_date=datetime(2023,1,1),
                                  end_date=datetime(2024,10,31))

    lead_time = random.randint(30, 90)
    expected_delivery = po_date + timedelta(days=lead_time)

    # Actual delivery — sometimes delayed
    delay_days = random.choices(
        [0, random.randint(1,7), random.randint(8,20), random.randint(21,45)],
        weights=[40, 30, 20, 10]
    )[0]
    actual_delivery = expected_delivery + timedelta(days=delay_days)

    qty_ordered  = random.randint(100, 1000)
    qty_received = int(qty_ordered * random.uniform(0.85, 1.0))  # partial shipments
    unit_cost    = sku['cost_price'] * random.uniform(0.95, 1.05)
    po_value     = round(qty_ordered * unit_cost, 2)

    po_records.append({
        'po_id':              f'PO{po_id:06d}',
        'sku_id':             sku['sku_id'],
        'brand':              sku['brand'],
        'category':           sku['category'],
        'supplier':           random.choice(suppliers),
        'po_date':            po_date.strftime('%Y-%m-%d'),
        'expected_delivery':  expected_delivery.strftime('%Y-%m-%d'),
        'actual_delivery':    actual_delivery.strftime('%Y-%m-%d'),
        'lead_time_days':     lead_time,
        'delay_days':         delay_days,
        'is_delayed':         1 if delay_days > 0 else 0,
        'qty_ordered':        qty_ordered,
        'qty_received':       qty_received,
        'fulfillment_rate':   round(qty_received/qty_ordered*100, 2),
        'unit_cost':          round(unit_cost, 2),
        'po_value':           po_value,
        'po_status':          'Delivered' if actual_delivery <= datetime(2024,12,31).date() else 'Pending'
    })
    po_id += 1

df_po = pd.DataFrame(po_records)
df_po.to_csv(os.path.join(RAW, 'purchase_orders.csv'), index=False)

print(f"Purchase Orders table: {df_po.shape[0]:,} rows × {df_po.shape[1]} columns ✅")
df_po.head(3)

Purchase Orders table: 8,000 rows × 17 columns ✅


,po_id,sku_id,brand,category,supplier,po_date,expected_delivery,actual_delivery,lead_time_days,delay_days,is_delayed,qty_ordered,qty_received,fulfillment_rate,unit_cost,po_value,po_status
0,PO000001,SKU01082,Bewakoof,Footwear,Mumbai Exports Ltd,2023-10-09,2023-12-30,2024-01-06,82,7,1,743,721,97.04,751.95,558699.74,Delivered
1,PO000002,SKU01458,Urbanic,Accessories,Bangalore Weaves,2023-05-25,2023-06-27,2023-06-27,33,0,0,788,739,93.78,491.82,387554.75,Delivered
2,PO000003,SKU01165,Snitch,Ethnic,Bangalore Weaves,2023-01-12,2023-02-26,2023-02-26,45,0,0,145,133,91.72,1022.18,148216.72,Delivered


In [12]:
platforms = ['Instagram', 'Pinterest', 'YouTube', 'Meesho Social', 'Moj', 'ShareChat']

social_records = []
sig_id = 1

for trend in trend_names:
    trend_id = f'TRD{trend_names.index(trend)+1:03d}'

    # Get peak week for this trend from df_trends
    peak_wk = df_trends[df_trends['trend_name'] == trend]['peak_week'].iloc[0]

    for week_num in range(104):
        week_date = start_date + timedelta(weeks=week_num)

        # Base trend score for this week
        base_score = df_trends[
            (df_trends['trend_name'] == trend) &
            (df_trends['week_number'] == week_num + 1)
        ]['trend_score'].iloc[0]

        for platform in platforms:
            # Platform multipliers — Instagram dominates Indian fashion
            plat_mult = {
                'Instagram': 1.0, 'Pinterest': 0.6, 'YouTube': 0.7,
                'Meesho Social': 0.5, 'Moj': 0.4, 'ShareChat': 0.35
            }[platform]

            # Tier 1 cities lead trend discovery
            for tier in [1, 2, 3]:
                tier_mult = {1: 1.0, 2: 0.65, 3: 0.35}[tier]

                mentions   = int(base_score * plat_mult * tier_mult * random.randint(1000, 8000))
                hashtag_reach = int(mentions * random.uniform(8, 25))
                saves      = int(mentions * random.uniform(0.1, 0.4))
                shares     = int(mentions * random.uniform(0.05, 0.2))
                engagement = round((saves + shares) / max(hashtag_reach, 1) * 100, 4)

                social_records.append({
                    'signal_id':       f'SIG{sig_id:07d}',
                    'trend_id':        trend_id,
                    'trend_name':      trend,
                    'platform':        platform,
                    'city_tier':       tier,
                    'week_start_date': week_date.strftime('%Y-%m-%d'),
                    'week_number':     week_num + 1,
                    'mentions':        mentions,
                    'hashtag_reach':   hashtag_reach,
                    'saves':           saves,
                    'shares':          shares,
                    'engagement_rate': engagement,
                    'trend_score':     base_score,
                    'is_peak_week':    1 if abs(week_num - peak_wk) <= 2 else 0
                })
                sig_id += 1

df_social = pd.DataFrame(social_records)
df_social.to_csv(os.path.join(RAW, 'social_signals.csv'), index=False)

print(f"Social Signals table: {df_social.shape[0]:,} rows × {df_social.shape[1]} columns ✅")
df_social.head(3)

Social Signals table: 37,440 rows × 14 columns ✅


,signal_id,trend_id,trend_name,platform,city_tier,week_start_date,week_number,mentions,hashtag_reach,saves,shares,engagement_rate,trend_score,is_peak_week
0,SIG0000001,TRD001,Cottagecore,Instagram,1,2023-01-01,1,153,2132,23,14,1.7355,0.0218,0
1,SIG0000002,TRD001,Cottagecore,Instagram,2,2023-01-01,1,113,1198,34,9,3.5893,0.0218,0
2,SIG0000003,TRD001,Cottagecore,Instagram,3,2023-01-01,1,58,1153,13,8,1.8213,0.0218,0


In [13]:
# Maps which SKUs are relevant to which trends
# This is the CORE joining table for our analysis

mapping_records = []

for _, trend_row in df_trends[df_trends['week_number'] == 1].iterrows():
    relevant_cats = trend_row['relevant_categories'].split('|')

    # Get SKUs in relevant categories
    relevant_skus = df_products[
        df_products['category'].isin(relevant_cats)
    ]['sku_id'].tolist()

    # Sample a subset — not every SKU maps to every trend
    sampled_skus = random.sample(relevant_skus, min(len(relevant_skus), 60))

    for sku_id in sampled_skus:
        sku_row = df_products[df_products['sku_id'] == sku_id].iloc[0]

        # Relevance score — how well does this SKU fit this trend?
        relevance = round(random.uniform(0.4, 1.0), 2)

        mapping_records.append({
            'mapping_id':       f'MAP{len(mapping_records)+1:06d}',
            'trend_id':         trend_row['trend_id'],
            'trend_name':       trend_row['trend_name'],
            'sku_id':           sku_id,
            'brand':            sku_row['brand'],
            'category':         sku_row['category'],
            'subcategory':      sku_row['subcategory'],
            'relevance_score':  relevance,
            'is_hero_sku':      1 if relevance >= 0.85 else 0
        })

df_mapping = pd.DataFrame(mapping_records)
df_mapping.to_csv(os.path.join(RAW, 'trend_sku_mapping.csv'), index=False)

print(f"Trend-SKU Mapping table: {df_mapping.shape[0]:,} rows × {df_mapping.shape[1]} columns ✅")
df_mapping.head(3)

Trend-SKU Mapping table: 1,200 rows × 9 columns ✅


,mapping_id,trend_id,trend_name,sku_id,brand,category,subcategory,relevance_score,is_hero_sku
0,MAP000001,TRD001,Cottagecore,SKU01688,Libas,Accessories,Bucket Hat,0.96,1
1,MAP000002,TRD001,Cottagecore,SKU02065,Styli,Topwear,Sweatshirt,0.55,0
2,MAP000003,TRD001,Cottagecore,SKU01471,FabAlley,Topwear,Crop Top,0.63,0


In [14]:
import os

files = {
    'products.csv':          df_products,
    'trend_signals.csv':     df_trends,
    'inventory.csv':         df_inventory,
    'sales_transactions.csv':df_sales,
    'purchase_orders.csv':   df_po,
    'social_signals.csv':    df_social,
    'trend_sku_mapping.csv': df_mapping
}

print("=" * 55)
print("  DATASET GENERATION COMPLETE — PHASE 1 SUMMARY")
print("=" * 55)

total_rows = 0
for fname, df in files.items():
    fpath = os.path.join(RAW, fname)
    fsize = os.path.getsize(fpath) / 1024
    total_rows += len(df)
    print(f"  {fname:<30} {len(df):>8,} rows  {fsize:>8.1f} KB")

print("-" * 55)
print(f"  {'TOTAL':<30} {total_rows:>8,} rows")
print("=" * 55)
print()
print("Tables saved to Data/Raw ✅")
print()
print("Schema Overview:")
print(f"  products          → Master SKU catalog")
print(f"  trend_signals     → Weekly trend scores (2 years)")
print(f"  inventory         → Weekly stock snapshots per SKU")
print(f"  sales_transactions→ 120K individual sale records")
print(f"  purchase_orders   → PO history with delays")
print(f"  social_signals    → Platform-wise trend mentions")
print(f"  trend_sku_mapping → Trend ↔ SKU relevance bridge")

  DATASET GENERATION COMPLETE — PHASE 1 SUMMARY
  products.csv                      1,170 rows     109.4 KB
  trend_signals.csv                 2,080 rows     224.6 KB
  inventory.csv                   122,850 rows   11716.2 KB
  sales_transactions.csv          120,000 rows   17590.4 KB
  purchase_orders.csv               8,000 rows    1058.3 KB
  social_signals.csv               37,440 rows    3278.2 KB
  trend_sku_mapping.csv             1,200 rows      90.7 KB
-------------------------------------------------------
  TOTAL                           292,740 rows

Tables saved to Data/Raw ✅

Schema Overview:
  products          → Master SKU catalog
  trend_signals     → Weekly trend scores (2 years)
  inventory         → Weekly stock snapshots per SKU
  sales_transactions→ 120K individual sale records
  purchase_orders   → PO history with delays
  social_signals    → Platform-wise trend mentions
  trend_sku_mapping → Trend ↔ SKU relevance bridge


In [15]:
# ── Fix: Inject Realistic Stockouts into Inventory ────────────
import pandas as pd
import numpy as np
import os
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

RAW = r"D:\Projects\End-to-end projects\11. Fashion Trend Inventory\Data\Raw"

# Reload inventory
df_inventory = pd.read_csv(os.path.join(RAW, 'inventory.csv'))

np.random.seed(42)

# Select 15% of SKUs to experience stockout periods
all_skus = df_inventory['sku_id'].unique()
stockout_skus = np.random.choice(all_skus, size=int(len(all_skus)*0.15), replace=False)

stockout_count = 0

for sku in stockout_skus:
    sku_mask = df_inventory['sku_id'] == sku
    sku_indices = df_inventory[sku_mask].index.tolist()

    # Pick 1-3 random stockout windows per SKU
    num_windows = np.random.randint(1, 4)

    for _ in range(num_windows):
        # Random start point (not first or last 10 weeks)
        start_idx = np.random.randint(10, len(sku_indices) - 15)
        duration  = np.random.randint(2, 6)  # 2-5 weeks of stockout

        for w in range(duration):
            idx = sku_indices[start_idx + w]
            df_inventory.at[idx, 'closing_stock']   = 0
            df_inventory.at[idx, 'opening_stock']   = 0 if w > 0 else df_inventory.at[idx, 'opening_stock']
            df_inventory.at[idx, 'units_sold']      = 0
            df_inventory.at[idx, 'stockout_flag']   = 1
            df_inventory.at[idx, 'inventory_health']= 'Critical'
            df_inventory.at[idx, 'weeks_of_cover']  = 0.0
            df_inventory.at[idx, 'inventory_value'] = 0.0
            stockout_count += 1

# Save updated inventory
df_inventory.to_csv(os.path.join(RAW, 'inventory.csv'), index=False)

print(f"Stockout records injected: {stockout_count:,}")
print(f"SKUs with stockouts:       {len(stockout_skus):,}")
print(f"Total stockout_flag = 1:   {df_inventory['stockout_flag'].sum():,}")
print("Inventory CSV updated ✅")

Stockout records injected: 1,214
SKUs with stockouts:       175
Total stockout_flag = 1:   1,177
Inventory CSV updated ✅


In [16]:
# ── Push Updated Inventory to PostgreSQL ──────────────────────
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

DB_USER     = "postgres"
DB_PASSWORD = quote_plus(os.environ["DB_PASSWORD"])
DB_HOST     = "localhost"
DB_PORT     = "5432"
DB_NAME     = "fashion_trend_db"

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

# Drop and reload inventory only
with engine.connect() as conn:
    conn.execute(text("TRUNCATE TABLE inventory;"))
    conn.commit()
    print("Old inventory cleared ✅")

df_inventory.to_sql(
    name      = 'inventory',
    con       = engine,
    if_exists = 'append',
    index     = False,
    method    = 'multi',
    chunksize = 1000
)

print(f"Updated inventory loaded: {len(df_inventory):,} rows ✅")

# Verify
with engine.connect() as conn:
    result = conn.execute(text("SELECT COUNT(*) FROM inventory WHERE stockout_flag = 1"))
    print(f"Stockout records in PostgreSQL: {result.fetchone()[0]:,} ✅")

Old inventory cleared ✅
Updated inventory loaded: 122,850 rows ✅
Stockout records in PostgreSQL: 1,177 ✅
